# AI 桌面宠物客户端 - 演示笔记本

本笔记本演示 AI 桌面宠物客户端应用的核心功能。

## 目录
1. [概述](#概述)
2. [客户端-服务器通信](#客户端-服务器通信)
3. [Live2D 模型集成](#Live2D-模型集成)
4. [WSL2 命令执行](#WSL2-命令执行)
5. [模式切换](#模式切换)

## 概述

AI 桌面宠物客户端使用以下技术构建：
- **Electron**: 桌面应用框架
- **Vue.js**: 前端框架
- **Live2D**: 2D角色动画
- **Express**: 用于WSL2命令的本地服务器

In [ ]:
import requests
import json

# 配置
SERVER_URL = 'http://localhost:8000'
LOCAL_SERVER_URL = 'http://localhost:3000'

print(f"服务器地址: {SERVER_URL}")
print(f"本地服务器地址: {LOCAL_SERVER_URL}")

## 客户端-服务器通信

### 创建会话

首先，我们需要创建一个会话来维持对话上下文。

In [ ]:
# 创建新会话
response = requests.post(f"{SERVER_URL}/api/v1/session")
session_id = response.json()['session_id']

print(f"会话ID: {session_id}")

### 发送聊天消息

现在让我们向AI发送一条消息并获取响应。

In [ ]:
# 发送聊天消息
message = "你好，请介绍一下你自己！"

response = requests.post(
    f"{SERVER_URL}/api/v1/chat",
    json={
        "session_id": session_id,
        "message": message,
        "stream": False
    }
)

data = response.json()
print(f"AI响应: {data.get('text', '无响应')}")
print(f"情感: {data.get('emotion', '未知')}")
print(f"动作: {data.get('action', 'idle')}")

## Live2D 模型集成

客户端使用 Live2D 模型来显示动画角色。以下是情感映射的工作原理：

In [ ]:
# 情感到Live2D的映射
emotion_map = {
    'happy': {'expression': 'exp_01', 'motion': 'special_01'},
    'sad': {'expression': 'exp_02', 'motion': 'mtn_02'},
    'angry': {'expression': 'exp_03', 'motion': 'mtn_03'},
    'playful': {'expression': 'exp_04', 'motion': 'special_02'},
    'shy': {'expression': 'exp_06', 'motion': 'mtn_01'},
    'neutral': {'expression': 'exp_01', 'motion': 'mtn_01'}
}

print("情感映射:")
for emotion, config in emotion_map.items():
    print(f"  {emotion}: 表情={config['expression']}, 动作={config['motion']}")

## WSL2 命令执行

客户端使用本地 Express 服务器来安全地执行 WSL2 命令。

In [ ]:
# 通过本地服务器执行WSL2命令
command = "ls -la"
timeout = 30

try:
    response = requests.post(
        f"{LOCAL_SERVER_URL}/execute",
        json={
            "command": command,
            "timeout": timeout
        }
    )
    
    result = response.json()
    print(f"命令: {result['command']}")
    print(f"退出代码: {result['returncode']}")
    print(f"输出:\n{result['stdout']}")
    if result['stderr']:
        print(f"错误:\n{result['stderr']}")
        
except requests.exceptions.ConnectionError:
    print("本地服务器未运行。请先启动它。")

## 模式切换

客户端支持两种主要模式：
1. **聊天模式**: 与AI进行正常对话
2. **WSL2模式**: 通过AI执行WSL2命令

In [ ]:
# 示例：聊天模式
print("=== 聊天模式 ===")
response = requests.post(
    f"{SERVER_URL}/api/v1/chat",
    json={
        "session_id": session_id,
        "message": "今天天气怎么样？",
        "mode": "chat",
        "stream": False
    }
)
data = response.json()
print(f"响应: {data.get('text', '无响应')}")

# 示例：WSL2模式
print("\n=== WSL2模式 ===")
response = requests.post(
    f"{SERVER_URL}/api/v1/chat",
    json={
        "session_id": session_id,
        "message": "列出当前目录的文件",
        "mode": "wsl2",
        "stream": False
    }
)
data = response.json()
print(f"WSL2命令: {data.get('wsl2_command', '无命令')}")
print(f"响应: {data.get('text', '无响应')}")

## 总结

本笔记本演示了：
- 创建会话和发送聊天消息
- 理解情感到Live2D的映射
- 通过本地服务器执行WSL2命令
- 在聊天和WSL2模式之间切换

更多信息请参考项目文档。